## Week 4 - Assignment 4 for COfDSAI Tutorial Markdown ##

This week we look to classify handwritten numbers to help a postal company know how accurate their software will be.

1. Setup, Data Splitting, and SVM Helper Functions

Start by configuring the experiment parameters, loading the MNIST dataset, and spliting the 60,000 training images into three sets:
- Validation Set A (`n_vala = 5,000`): Used to compare kernels and tune hyperparameters ($\gamma$ and $C$).
- Validation Set B (`n_valb = 10,000`): Held out untouched until the end to estimate out-of-sample error before touching the test set.
- Training Pool (`rest = 45,000`): Used to draw balanced training subsets for model selection (`n_grid = 500`) and the final classifier (`n_final = 4,000`).

It also defines the RBF/linear kernels, Cholesky factorization ($K + \varepsilon I = LL^\top$), CVXPY primal and dual solvers for the squared-hinge SVM, primal weight recovery, and disk caching.

In [ ]:
import os
import time
import pickle
import hashlib
import numpy as np
import scipy.linalg as sl
import cvxpy as cp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier

from load_mnist import load_mnist, subsample, confusion, key_numbers, show_digits

# 1. CONFIGURATION
STUDENT = "i6194134"
SOLVER = "CLARABEL"
EPS = 1e-6                                  # Jitter for positive-definite kernel matrix: K_eps = K + EPS * I

n_final = 4000                              # Training images for the final classifier (400 per digit)
n_grid = 500                                # Training images for model selection (50 per digit)
n_vala = 5000                               # Validation set A: hyperparameter tuning
n_valb = 10000                              # Validation set B: unbiased pre-test validation error

NG, NF = n_grid, n_final
NS = [NF // 8, NF // 4, NF // 2, NF]        # Learning-curve sample sizes: [500, 1000, 2000, 4000]

GAMMAS = [0.005, 0.01, 0.02, 0.04, 0.08, 0.16]   # Coarse RBF gamma grid
CS_RBF = [1, 10, 100]                            # Coarse RBF C grid
CS_LIN = [0.001, 0.01, 0.1, 1]                   # Linear kernel C grid
CS_SWEEP = [0.1, 0.3, 1, 3, 10, 30, 100, 300]    # Fine C-sweep at the best gamma

SEED_SPLIT = 1                              # Seed for train/valA/valB split
SEED_TRAIN = 0                              # Seed for balanced subsampling from training pool

os.makedirs("cache", exist_ok=True)

# 2. DATA LOADING & DISJOINT SPLITTING
X, y, Xt, yt = load_mnist()

# Partition the 60,000 training images into disjoint sets: valA, valB, and rest
pool = np.random.default_rng(SEED_SPLIT).permutation(len(X))
valA = pool[:n_vala]
valB = pool[n_vala : n_vala + n_valb]
rest = pool[n_vala + n_valb :]

assert not (set(valA) & set(valB)) and not (set(rest) & (set(valA) | set(valB)))


# 3. KERNEL, SQUARED HINGE LOSS, and SVM SOLVERS (primal and dual)
def sqdist(A, B):
    # Pairwise squared Euclidean distance matrix between rows of A and B
    return np.maximum((A * A).sum(1)[:, None] + (B * B).sum(1)[None, :] - 2 * A @ B.T, 0.0)


def kernel(A, B, kind="rbf", gamma=0.02):
    # Compute linear (A B^T) or RBF (exp(-gamma * ||a - b||^2)) kernel matrix
    if kind == "linear":
        return A @ B.T
    return np.exp(-gamma * sqdist(A, B))


def factor(K):
    # Add small jitter EPS*I and compute the lower-triangular Cholesky factor L (K_eps = L L^T)
    Keps = K + EPS * np.eye(len(K))
    return Keps, np.linalg.cholesky(Keps)


def solve_primal(L, y, C):
    # Solve the squared-hinge SVM primal QP in CVXPY
    n = len(y)
    u = cp.Variable(n)
    b = cp.Variable()
    xi = cp.Variable(n)

    objective = cp.Minimize(0.5 * cp.sum_squares(u) + C * cp.sum_squares(xi))
    constraints = [cp.multiply(y, L @ u + b) >= 1 - xi]
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=SOLVER)
    return u.value, float(b.value), float(prob.value), prob.status


def dual_problem(L, C):
    # Build the squared-hinge SVM Lagrange dual QP once using y as a CVXPY Parameter (DPP)
    n = L.shape[0]
    a = cp.Variable(n)
    yp = cp.Parameter(n)
    obj = cp.sum(a) - 0.5 * cp.sum_squares(L.T @ cp.multiply(yp, a)) - cp.sum_squares(a) / (4 * C)
    prob = cp.Problem(cp.Maximize(obj), [a >= 0, yp @ a == 0])
    return prob, a, yp


def solve_dual(prob, a, yp, y):
    # Update the label parameter y and solve the compiled dual problem
    yp.value = y
    prob.solve(solver=SOLVER)
    return np.maximum(a.value, 0.0), float(prob.value), prob.status


def recover(Keps, y, alpha, C):
    # Recover kernel weights c = alpha o y and intercept b from active dual variables (alpha_i > 0)
    c = alpha * y
    u_f = Keps @ c                                  # Unshifted decision scores (= L u)
    sv = alpha > 1e-8 * max(alpha.max(), 1e-300)    # Active support vectors
    b = np.mean(y[sv] * (1 - alpha[sv] / (2 * C)) - u_f[sv]) if sv.any() else 0.0
    return c, b


def fit_ovr_dual(Keps, L, ylab, C):
    # Fit 10 one-vs-rest binary dual SVMs sharing one kernel matrix and one compiled CVXPY problem
    prob, a, yp = dual_problem(L, C)
    cs, bs = [], []
    for k in range(10):
        yk = np.where(ylab == k, 1.0, -1.0)
        alpha, _, _ = solve_dual(prob, a, yp, yk)
        c, b = recover(Keps, yk, alpha, C)
        cs.append(c)
        bs.append(b)
    return np.array(cs), np.array(bs)


def solve_binary(Keps, L, y, C):
    # Solve both primal and dual for a single binary digit problem and record timing/duality checks
    out = {}

    t0 = time.perf_counter()
    u, b_p, pv, ps = solve_primal(L, y, C)
    out["tp"] = time.perf_counter() - t0

    prob, a, yp = dual_problem(L, C)
    t0 = time.perf_counter()
    alpha, dv, ds = solve_dual(prob, a, yp, y)
    out["td"] = time.perf_counter() - t0

    c_d, b_d = recover(Keps, y, alpha, C)
    c_p = sl.solve_triangular(L.T, u, lower=False)  # Recover primal kernel weights: c = L^{-T} u

    out.update(
        pv=pv, dv=dv, gap=abs(pv - dv),
        alpha=alpha, c_p=c_p, b_p=b_p, c_d=c_d, b_d=b_d,
        status=(ps, ds),
    )
    return out


# 4. SAMPLING, CACHING, AND EVALUATION HELPERS
def draw(n):
    # Draw a balanced, nested subset of size n from the training pool
    return rest[subsample(y[rest], n, SEED_TRAIN)]


def cache(name, key, fn):
    # Load cached result from ./cache/ if key matches; otherwise compute, save, and return
    f = "cache/%s_%s.pkl" % (name, hashlib.md5(repr(key).encode()).hexdigest()[:12])
    if os.path.exists(f):
        return pickle.load(open(f, "rb"))
    res = fn()
    pickle.dump(res, open(f, "wb"))
    return res


def err(pred, true):
    # Classification error rate in percent
    return 100.0 * np.mean(pred != true)


def se(p, n):
    # Binomial standard error of a percentage error rate p over n samples (in percentage points)
    return 100.0 * np.sqrt((p / 100.0) * (1.0 - p / 100.0) / n)


def predict(cs, bs, Xs, Xnew, kind, gamma):
    # Predict digit labels via argmax_k (sum_i c_i^(k) k(x_i, x) + b^(k)
    return np.argmax(kernel(Xnew, Xs, kind, gamma) @ cs.T + bs, axis=1)


def fit_ovr(idx, kind, gamma, C):
    # Factorize kernel matrix once and fit all 10 OvR dual SVMs
    Xs, ys = X[idx], y[idx]
    Keps, L = factor(kernel(Xs, Xs, kind, gamma))
    return fit_ovr_dual(Keps, L, ys, C)


def fit_eval(n, kind, gamma, C):
    # Fit one OvR model on n training images and return (train_err %, valA_err %, elapsed_sec)
    key = (n, kind, gamma, C, SEED_SPLIT, SEED_TRAIN, n_vala, SOLVER)

    def run():
        idx = draw(n)
        t0 = time.time()
        cs, bs = fit_ovr(idx, kind, gamma, C)
        elapsed = time.time() - t0
        tr_e = err(predict(cs, bs, X[idx], X[idx], kind, gamma), y[idx])
        va_e = err(predict(cs, bs, X[idx], X[valA], kind, gamma), y[valA])
        return (tr_e, va_e, elapsed)

    r = cache("fit", key, run)
    print("fit %s -> train %.2f%% | valA %.2f%% (%.0fs)" % (key[:4], r[0], r[1], r[2]), flush=True)
    return r

2. Model Selection, Reference Baselines, and Learning Curve

Using `n_grid = 500` training images scored strictly on Validation Set A, this cell:
- Linear vs. RBF comparison: Evaluates a linear kernel across `CS_LIN` and an RBF kernel across a coarse $\gamma \times C$ grid (`GAMMAS x CS_RBF`) to pick the best $\gamma$.
- Fine $C$-sweep: Fixes the best $\gamma$ and sweeps `CS_SWEEP` to select the final regularization parameter $C$, while computing $\pm 1$ standard error bands to see which values are statistically indistinguishable.
- Reference benchmarks: Fits scikit-learn's standard hinge-loss RBF SVM (both OvR and OvO) at the same $(\gamma, C)$ for comparison.
- Learning curve: Evaluates validation-A error and solve time across increasing training sizes `NS = [500, 1000, 2000, 4000]`.

In [ ]:
# --- 1. Linear baseline & coarse RBF (gamma x C) grid search ---
lin = {C: fit_eval(NG, "linear", 0, C) for C in CS_LIN}
bestlin = min(lin.items(), key=lambda kv: kv[1][1])

grid = {(g, C): fit_eval(NG, "rbf", g, C) for g in GAMMAS for C in CS_RBF}
(GAMMA, CG), gbest = min(grid.items(), key=lambda kv: kv[1][1])

# Identify all gamma values within 1 standard error of the best validation-A error
gam_curve = {g: min(grid[(g, C)][1] for C in CS_RBF) for g in GAMMAS}
se_g = se(grid[(GAMMA, CG)][1], n_vala)
gam_ok = [g for g in GAMMAS if gam_curve[g] <= grid[(GAMMA, CG)][1] + se_g]

# --- 2. Fine C-sweep at the selected gamma ---
csweep = [(C,) + fit_eval(NG, "rbf", GAMMA, C)[:2] for C in CS_SWEEP]
CBEST, _, vC = min(csweep, key=lambda r: r[2])

# Identify all C values within 1 standard error of the minimum, and first C with 0% train error
se_c = se(vC, n_vala)
c_ok = [r[0] for r in csweep if r[2] <= vC + se_c]
c_zero = min((r[0] for r in csweep if r[1] == 0), default=None)

print(f"Best gamma: {GAMMA} | Best C: {CBEST}")
print(f"Gammas within 1 SE: {gam_ok} | Cs within 1 SE: {c_ok} | First C with 0% train err: {c_zero}")


# --- 3. Scikit-learn hinge-loss reference models (OvO and OvR) at the same (GAMMA, CBEST) ---
def stage_ref():
    idx = draw(NG)
    res = {}
    for name, model in (
        ("ovo", SVC(kernel="rbf", gamma=GAMMA, C=CBEST * 2)),
        ("ovr", OneVsRestClassifier(SVC(kernel="rbf", gamma=GAMMA, C=CBEST * 2))),
    ):
        model.fit(X[idx], y[idx])
        res[name] = err(model.predict(X[valA]), y[valA])
    return res


ref = cache("ref", (NG, GAMMA, CBEST, SEED_SPLIT, SEED_TRAIN, n_vala), stage_ref)
print(f"Reference hinge-loss SVMs on valA -> OvR: {ref['ovr']:.2f}% | OvO: {ref['ovo']:.2f}%")

# --- 4. Learning curve across increasing training sizes NS ---
ncurve = [(n,) + fit_eval(n, "rbf", GAMMA, CBEST) for n in NS]   # (n, train_err, valA_err, seconds)

fit (500, 'linear', 0, 0.001) -> train 11.80 val 19.26 (5s)


fit (500, 'linear', 0, 0.01) -> train 3.00 val 16.72 (5s)
fit (500, 'linear', 0, 0.1) -> train 0.00 val 18.38 (5s)
fit (500, 'linear', 0, 1) -> train 0.00 val 20.58 (7s)
fit (500, 'rbf', 0.005, 1) -> train 1.60 val 14.54 (4s)
fit (500, 'rbf', 0.005, 10) -> train 0.00 val 13.42 (5s)
fit (500, 'rbf', 0.005, 100) -> train 0.00 val 13.80 (5s)
fit (500, 'rbf', 0.01, 1) -> train 0.40 val 13.08 (3s)
fit (500, 'rbf', 0.01, 10) -> train 0.00 val 12.36 (5s)
fit (500, 'rbf', 0.01, 100) -> train 0.00 val 12.44 (5s)
fit (500, 'rbf', 0.02, 1) -> train 0.00 val 11.88 (4s)
fit (500, 'rbf', 0.02, 10) -> train 0.00 val 11.40 (4s)
fit (500, 'rbf', 0.02, 100) -> train 0.00 val 11.38 (4s)
fit (500, 'rbf', 0.04, 1) -> train 0.00 val 12.06 (4s)
fit (500, 'rbf', 0.04, 10) -> train 0.00 val 11.30 (4s)
fit (500, 'rbf', 0.04, 100) -> train 0.00 val 11.28 (4s)
fit (500, 'rbf', 0.08, 1) -> train 0.00 val 15.92 (3s)
fit (500, 'rbf', 0.08, 10) -> train 0.00 val 15.32 (4s)
fit (500, 'rbf', 0.08, 100) -> train 0.00 va

3. Final Classifier: Primal & Dual Verification and Evaluation

With $(\gamma, C)$ fixed, this cell trains the final 10-digit OvR classifier on `n_final = 4,000` images by solving both the primal and the dual** in CVXPY for every digit. It then:
- Verifies numerical optimality (solver status, largest absolute/relative duality gap, primal-vs-dual weight recovery error, and test prediction agreement).
- Evaluates the final classifier on the training set, Validation Set B (unbiased out-of-sample estimate), and the 10,000-image Test Set (new images).

In [ ]:
# --- 1. Solve primal and dual for all 10 binary digit problems at n = NF ---
def stage_final():
    idx = draw(NF)
    Xs, ys = X[idx], y[idx]

    # Factorize the shared kernel matrix once across all 10 digits
    t0 = time.perf_counter()
    Keps, L = factor(kernel(Xs, Xs, "rbf", GAMMA))
    tchol = time.perf_counter() - t0

    R = {"idx": idx, "tchol": tchol, "probs": []}
    for k in range(10):
        yk = np.where(ys == k, 1.0, -1.0)
        o = solve_binary(Keps, L, yk, CBEST)
        R["probs"].append(o)
        print("final digit %d | tp %.1fs | td %.1fs | gap %.2e" % (k, o["tp"], o["td"], o["gap"]), flush=True)
    return R


fin = cache("final", (NF, GAMMA, CBEST, SOLVER, EPS, SEED_TRAIN, SEED_SPLIT, n_vala, n_valb), stage_final)

# --- 2. Extract primal and dual parameters & generate predictions ---
idx = fin["idx"]
Xs, ys = X[idx], y[idx]
P = fin["probs"]

Cd = np.array([p["c_d"] for p in P])
bd = np.array([p["b_d"] for p in P])
Cp = np.array([p["c_p"] for p in P])
bp = np.array([p["b_p"] for p in P])

pred = lambda C_, b_, Z: predict(C_, b_, Xs, Z, "rbf", GAMMA)

tr_d = pred(Cd, bd, Xs)                         # Training predictions (dual)
vB_d = pred(Cd, bd, X[valB])                    # Validation-B predictions (dual)
te_d = pred(Cd, bd, Xt)                         # Test predictions (dual) — test set used ONCE
te_p = pred(Cp, bp, Xt)                         # Test predictions (primal) — for agreement check

# --- 3. Compute key metrics, confusion matrix, and numerical checks ---
kn_tr = key_numbers(ys, tr_d)
kn_val = key_numbers(y[valB], vB_d)
kn_te = key_numbers(yt, te_d)
M = confusion(yt, te_d)

agree = 100.0 * np.mean(te_d == te_p)
gap_max = max(p["gap"] for p in P)
gap_rel = max(p["gap"] / abs(p["pv"]) for p in P)
rec = max(np.linalg.norm(p["c_p"] - p["c_d"]) / np.linalg.norm(p["c_d"]) for p in P)
tp_tot = fin["tchol"] + sum(p["tp"] for p in P)
td_tot = sum(p["td"] for p in P)
status = all(p["status"] == ("optimal", "optimal") for p in P)

print(f"\nTrain: {kn_tr}")
print(f"Val B: {kn_val}")
print(f"Test : {kn_te}")
print(
    f"Checks -> agree: {agree:.2f}% | max gap: {gap_max:.2e} (rel {gap_rel:.2e}) "
    f"| weight rec err: {rec:.2e} | tp: {tp_tot:.1f}s | td: {td_tot:.1f}s | all optimal: {status}"
)

final digit 0 tp 139.7 td 143.3 gap 2.39e-07
final digit 1 tp 156.8 td 145.2 gap 1.82e-07
final digit 2 tp 155.1 td 131.6 gap 8.88e-07
final digit 3 tp 148.5 td 139.1 gap 1.97e-06
final digit 4 tp 169.0 td 148.6 gap 2.78e-07
final digit 5 tp 160.0 td 137.3 gap 1.84e-06
final digit 6 tp 165.0 td 143.3 gap 1.09e-07
final digit 7 tp 158.7 td 127.3 gap 1.24e-07
final digit 8 tp 129.8 td 109.2 gap 2.30e-06
final digit 9 tp 137.8 td 128.6 gap 1.14e-06
{'error': 0.0, 'confused_pair': '0-0', 'confused_n': 0} {'error': 4.51, 'confused_pair': '3-5', 'confused_n': 35} {'error': 4.01, 'confused_pair': '2-7', 'confused_n': 31} | agree 100.0 gap 2.302336923776238e-06 rec 1.3710317814597015e-05 tp 1521.6373035999713 td 1353.4195928000845 status True


4. Identifying Hard Digits from Dual Variables

Using the dual multipliers $\alpha_i^{(k)} = 2C\xi_i^{(k)}$ from the final classifier, this cell:
- Measures the share of training images that act as support vectors ($\alpha_i^{(k)} > 0$ for at least one digit) and how concentrated the total slack $\sum_k \alpha_i^{(k)} / (2C)$ is in the top 10% hardest images.
- Selects the 16 hardest training images (largest total slack) and the 16 most confidently misclassified test images (largest margin between the wrong predicted score and the true digit's score).

In [ ]:
# --- 1. Rank training images by total squared-hinge slack from dual variables ---
A = np.array([p["alpha"] for p in P])           # Shape (10, NF): alpha_i^(k) = 2C * xi_i^(k)
hard = A.sum(axis=0) / (2 * CBEST)              # Total slack per training image across all 10 digits
order = np.argsort(-hard)                       # Indices sorted from hardest to easiest

# Support vector concentration statistics
tol = 1e-8 * A.max()
frac_any = 100.0 * np.mean(A.max(axis=0) > tol) # % of training images with at least one alpha > 0
frac_zero = 100.0 - frac_any                    # % of irrelevant training images (alpha = 0 everywhere)
share10 = 100.0 * hard[order[: NF // 10]].sum() / hard.sum()

top = order[:16]                                # 16 hardest training images
top_mis = int(np.sum(tr_d[top] != ys[top]))

# --- 2. Find the 16 most confidently misclassified test images ---
Ft = kernel(Xt, Xs, "rbf", GAMMA) @ Cd.T + bd   # Decision scores on test set
wrong = np.where(te_d != yt)[0]
confidence_gap = Ft[wrong, te_d[wrong]] - Ft[wrong, yt[wrong]]
wtop = wrong[np.argsort(-confidence_gap)[:16]]

print(f"Support vectors: {frac_any:.1f}% | Irrelevant: {frac_zero:.1f}% | Top 10% slack share: {share10:.1f}%")
print(f"Misclassified among top 16 hardest training images: {top_mis}/16")

 5. Figures and `keynumbers.csv` Export

This cell generates all report figures (`fig_main.pdf`, `fig_grid.pdf`, `fig_hard.pdf`), prints the test confusion matrix, and writes the required `keynumbers.csv` file for submission.

In [ ]:
plt.rcParams.update({"font.size": 8})
c = np.array(csweep)
n_ = np.array(ncurve)

# 1. FIGURE 1 (fig_main.pdf): C-sweep | Learning curve | Test confusion matrix
fig, ax = plt.subplots(1, 3, figsize=(7.4, 1.95), gridspec_kw={"width_ratios": [1, 1, 1.05]})

# Panel A: Fine C-sweep at best gamma
ax[0].semilogx(c[:, 0], c[:, 1], "o-", label="train")
ax[0].semilogx(c[:, 0], c[:, 2], "s-", label="validation")
ax[0].axvline(CBEST, color="k", ls="--", lw=0.8)
ax[0].plot([CBEST], [vC], "k*", ms=9, zorder=5)
ax[0].annotate(
    f"chosen $C$={CBEST:g}\n(val. {vC:.1f}%)",
    (CBEST, vC),
    xytext=(c[0, 0] * 1.2, vC * 0.62),
    fontsize=7,
    arrowprops=dict(arrowstyle="->", lw=0.6),
)
ax[0].set_xlabel(f"$C$ (n={NG}, $\\gamma$={GAMMA:g})")
ax[0].set_ylabel("error (%)")
ax[0].legend(frameon=False, fontsize=7, loc="upper center")
ax[0].grid(alpha=0.3)

# Panel B: Learning curve (error vs. training size n)
ax[1].semilogx(n_[:, 0], n_[:, 1], "o-", label="train")
ax[1].semilogx(n_[:, 0], n_[:, 2], "s-", label="validation")
ax[1].set_xticks(NS)
ax[1].set_xticklabels([str(v) for v in NS])
ax[1].minorticks_off()
ax[1].set_xlabel(f"training images $n$ ($C$={CBEST:g})")
ax[1].legend(frameon=False)
ax[1].grid(alpha=0.3)

# Panel C: Off-diagonal test confusion matrix (masked diagonal for clear color scaling)
M_off = M.copy().astype(float)
np.fill_diagonal(M_off, np.nan)
vmax_off = np.nanmax(M_off)

cmap = plt.cm.Blues.copy()
cmap.set_bad(color="#1f3b73")
ax[2].imshow(M_off, cmap=cmap, vmin=0, vmax=vmax_off)
for i in range(10):
    for j in range(10):
        if i != j and M[i, j] > 0:
            ax[2].text(
                j, i, M[i, j],
                ha="center", va="center", fontsize=5.5,
                color="w" if M[i, j] > 0.55 * vmax_off else "k",
            )
ax[2].set_xticks(range(10))
ax[2].set_yticks(range(10))
ax[2].set_xlabel("predicted")
ax[2].set_ylabel("true")
ax[2].set_title("test confusion (off-diagonal)", fontsize=7)

fig.tight_layout(pad=0.3)
fig.savefig("fig_main.pdf")
plt.close(fig)

# 2. FIGURE 2 (fig_grid.pdf): Model-selection curves (gamma grid & C-sweep)
fig, ax = plt.subplots(1, 2, figsize=(7.4, 2.5))

for C_val in CS_RBF:
    ax[0].semilogx(GAMMAS, [grid[(g, C_val)][1] for g in GAMMAS], "o-", label=f"RBF, $C$={C_val:g}")
ax[0].axhline(bestlin[1][1], color="gray", ls="--", label=f"best linear ($C$={bestlin[0]:g})")
ax[0].axhspan(grid[(GAMMA, CG)][1] - se_g, grid[(GAMMA, CG)][1] + se_g, color="k", alpha=0.08, label="best $\\pm$1 SE")
ax[0].axvline(GAMMA, color="k", ls=":", lw=0.8)
ax[0].set_xlabel(f"$\\gamma$ (n={NG})")
ax[0].set_ylabel("validation-A error (%)")
ax[0].legend(frameon=False, fontsize=6.5)
ax[0].grid(alpha=0.3)

ax[1].semilogx(c[:, 0], c[:, 1], "o-", label="train")
ax[1].semilogx(c[:, 0], c[:, 2], "s-", label="validation")
ax[1].axhspan(vC - se_c, vC + se_c, color="k", alpha=0.08, label="best $\\pm$1 SE")
ax[1].axvline(CBEST, color="k", ls=":", lw=0.8)
ax[1].set_title("train and validation error at best $\\gamma$", fontsize=7)
ax[1].set_xlabel(f"$C$ ($\\gamma$={GAMMA:g})")
ax[1].legend(frameon=False, fontsize=6.5)
ax[1].grid(alpha=0.3)

fig.tight_layout(pad=0.4)
fig.savefig("fig_grid.pdf")
plt.close(fig)

# 3. FIGURE 3 (fig_hard.pdf): Hardest training & most confidently wrong test digits
imgs = np.vstack([Xs[top], Xt[wtop]])
titles = [f"{ys[i]}→{tr_d[i]}" for i in top] + [f"{yt[i]}→{te_d[i]}" for i in wtop]
show_digits(imgs, titles=titles, ncols=16, size=0.62).savefig("fig_hard.pdf")
plt.close("all")

# 4. EXPORT keynumbers.csv (for Appendix A)
key_rows = [
    ("student_number", STUDENT),
    ("model", f"OvR squared-hinge SVM; RBF kernel with gamma={GAMMA:g} and C={CBEST:g}; chosen on {n_vala} validation images; solved via primal and dual"),
    ("loss", "other"),
    ("multiclass", "ovr"),
    ("kernel", "rbf"),
    ("n_train", NF),
    ("train_error", f"{kn_tr['error']:.2f}"),
    ("val_error", f"{kn_val['error']:.2f}"),
    ("n_test", len(yt)),
    ("test_error", f"{kn_te['error']:.2f}"),
    ("confused_pair", kn_te["confused_pair"]),
    ("duality_gap", f"{gap_max:.1e}"),
    ("time_primal", f"{tp_tot:.1f}"),
    ("time_dual", f"{td_tot:.1f}"),
]

with open("keynumbers.csv", "w") as f:
    f.write("key,value\n")
    for k, v in key_rows:
        f.write(f"{k},{v}\n")

print("Done. Wrote keynumbers.csv, fig_main.pdf, fig_grid.pdf, fig_hard.pdf")
print("Test confusion matrix:\n", M)


Done. Wrote results.tex, keynumbers.csv, fig_main.pdf, fig_grid.pdf, fig_hard.pdf
test confusion matrix:
 [[ 967    0    2    0    0    2    6    1    2    0]
 [   0 1124    2    1    0    2    5    0    1    0]
 [  15    0  974    8    3    0    2   13   16    1]
 [   0    0    9  955    1   11    3    8   19    4]
 [   1    1    3    0  949    0    7    2    3   16]
 [   5    0    3   17    1  846    8    2    4    6]
 [  11    3    1    0    2    6  934    0    1    0]
 [   0   15   18    0    6    0    0  979    1    9]
 [   5    1    5    7    7    8    4   11  924    2]
 [   9    4    1    9   14    6    2    8    9  947]]
